In [5]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import ollama
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

In [65]:
load_dotenv(override=True)

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter_api_key = os.environ["OPENROUTER_API_KEY"]
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

ollama_client = ollama.Client(host="http://127.0.0.1:11434")

In [15]:
def find_repo_root(start: Path = None) -> Path:
    """Walk up from start (default: cwd) until a folder containing pyproject.toml is found."""
    start = start or Path.cwd()
    for folder in [start, *start.parents]:
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above " + str(start))

DATA = find_repo_root() / "data"

In [18]:

def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))

def get_sample_data(tiny=False, held_out=False):
    """
    Return a list of claims, and the handler's decision on them keyed by claim id.

    held_out=True gives the twenty-five held out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")

    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decisions for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [19]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
claims = [claim for claim in claims if labels[claim["claim_id"]]["difficulty"] != "manual"]
policies = {policy["policy_id"]: policy for policy in load("policies.json")}

print(f"{len(claims)} claims")

20 claims


In [24]:
decisions = pd.DataFrame([labels[claim["claim_id"]] for claim in claims])
decisions["decision"].value_counts()

decision
approve    12
decline     5
refer       3
Name: count, dtype: int64

In [25]:
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")
short_wording = (DATA / "policy_wording_short.md").read_text(encoding="utf-8")

print(f"The policy wording is {len(policy_wording.split()):,} words")
print(f"The short version is {len(short_wording.split()):,} words")

The policy wording is 1,369 words
The short version is 808 words


In [27]:
def split_sections(wording):
    """Split the wording into its opening lines and a dict of its sections, keyed by heading."""
    parts = wording.split("\n## ")
    sections = {part.split("\n")[0].strip(): "## " + part for part in parts[1:]}
    return parts[0], sections

opening, sections = split_sections(short_wording)
list(sections)

['Limits',
 'Cancellation and Interruption',
 'Trip Delay',
 'Baggage Delay',
 'Baggage',
 'Passport',
 'Emergency Medical',
 'Add-ons',
 'Never covered',
 'Payment',
 'Claims']

In [28]:
BENEFITS = {
    "trip_cancellation": "Cancellation and Interruption",
    "trip_interruption": "Cancellation and Interruption",
    "trip_delay": "Trip Delay",
    "baggage_delay": "Baggage Delay",
    "baggage": "Baggage",
    "passport": "Passport",
    "emergency_medical": "Emergency Medical",
}


def make_wording(claim):
    """The sections of the short wording that this claim needs: the limits, its benefit, the add-ons if the policy has any, and the rules for every claim."""
    headings = ["Limits", BENEFITS[claim["claim_type"]]]
    if policies[claim["policy_id"]]["add_ons"]:
        headings.append("Add-ons")
    headings += ["Never covered", "Payment", "Claims"]
    return "\n\n".join([opening.strip()] + [sections[heading].strip() for heading in headings])

In [29]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using the Payment section.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""


def make_user_prompt(claim):
    """The policy record and the claim as compact JSON, without the ids and the text of the documents."""
    policy = policies[claim["policy_id"]]
    record = {key: value for key, value in policy.items() if key not in ["policy_id", "customer_id", "sold_through", "status"]}
    details = {key: value for key, value in claim.items() if key not in ["policy_id", "customer_id", "status"]}
    details["documents"] = [{"name": document["name"], "type": document["type"]} for document in claim["documents"]]
    return f"""# Policy record

{json.dumps(record)}

# Claim

{json.dumps(details)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording it needs, then the policy record and the claim."""
    return [
        {"role": "system", "content": f"{INSTRUCTIONS}\n{make_wording(claim)}\n"},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

In [30]:
messages = make_messages(claims[0])
print(messages[0]["content"])
print(messages[1]["content"])

You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using the Payment section.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

# UnderCover Travel Insurance: Policy Wording, short version

All amounts are in US dollars. The policy record shows the plan, the add-ons, the trip dates, the trip cost and the travelers.

## Limits

| Benefit | Economy | Premium | First |
|---|---|---|---|
| Cancellation and Interruption maximum | 1,500 | 5,000 | 10,000 |
| Trip Delay waiting period, hours | 12 | 6 | 3 |
| Trip Delay maximum | 300 | 800 | 1,500 |
| Baggage Delay waiting period, hours | 24 | 12 | 6 |
| Bag

In [31]:
class ClaimDecision(BaseModel):
    """A claim handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or - for a decline or a referral")


In [36]:
def decide(claim, model, effort, ignore=None):
    """Ask one model OpenRouter to decide on one claim, skipping any providers listed in the ignore."""
    start = time.time()
    # tells OpenRouter to only use a model that supports all requirements in this request (e.g. structured outputs)
    provider = {"require_parameters": True}

    if ignore:
        provider["ignore"] = ignore

    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        reasoning_effort=effort,
        extra_body={"provider": provider},
    )

    answer = response.choices[0].message.parsed
    decision, payout, rationale = answer.decision, answer.payout, answer.rationale

    return {
        "model": model,
        "effort": effort,
        "where": "OpenRouter",
        "claim_id": claim["claim_id"],
        "decision": decision,
        "payout": payout,
        "rationale": rationale,
        "provider": response.provider,
        "input_tokens": response.usage.prompt_tokens,
        "cost": response.usage.cost,
        "seconds": round(time.time()- start, 1),
    }

In [39]:
first = decide(claims[0], "deepseek/deepseek-v4.1-flash", "low")
first

{'model': 'deepseek/deepseek-v4.1-flash',
 'effort': 'low',
 'where': 'OpenRouter',
 'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'rationale': 'The claim is covered under Trip Delay. The flight home was canceled by the carrier due to a crew problem, and the traveler was rebooked the next morning, exceeding the Premium plan’s 6-hour waiting period. The claimed items are covered trip-delay expenses: accommodation $245, meals $95, and local transport/taxis $80 = $420. All required receipts and the carrier cancellation notice are attached. No carrier reimbursement or deductible applies, and the amount is below the Premium Trip Delay maximum of $800.',
 'provider': 'Wafer',
 'input_tokens': 1055,
 'cost': 0.000586039,
 'seconds': 5.8}

In [44]:
def make_day2_messages(claim):
    """Day 2's prompt: the whole policy wording, and the policy record and the claim as indented JSON"""
    policy = policies[claim["policy_id"]]

    return [
        {"role": "system", "content": f"{INSTRUCTIONS}\n{policy_wording}\n"},
        {"role": "user", "content": f"# Policy record\n\n{json.dumps(policy, indent=2)}\n\n# Claim\n\n{json.dumps(claim, indent=2)}"},
    ]


day2 = openrouter.chat.completions.parse(
    model="deepseek/deepseek-v4.1-flash",
    messages=make_day2_messages(claims[0]),
    response_format=ClaimDecision,
    reasoning_effort="low",
    extra_body={"provider": {"require_parameters": True}},
)

print(f"Day 2's prompt: {day2.usage.prompt_tokens:,} input tokens")
print(f"Today's prompt: {first['input_tokens']:,} input tokens")

Day 2's prompt: 2,892 input tokens
Today's prompt: 1,055 input tokens


In [45]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def score(answer, label):
    """Add the difficulty, and whether the decision, and the decision with the payout, match the handler's."""
    return {
        **answer,
        "difficulty": label["difficulty"],
        "decision_right": answer["decision"] == label["decision"],
        "right": is_right(answer, label),
    }


def report(results):
    """Print how a run did: the decisions right, the decisions right with the payout too, the cost and the time."""
    decisions_right = sum(result["decision_right"] for result in results)
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{results[0]['model']}: {decisions_right} of {len(results)} decisions right, {right} with the payout too, ${cost:.4f} in {seconds:.0f} seconds")

In [49]:
def run(model, effort, claims, labels, ignore=None):
    """Run a model on OpenRouter on the claims and compare against the labels."""
    results = []
    for claim in tqdm(claims, desc=model):
        answer = decide(claim, model, effort, ignore)
        results.append(score(answer, labels[claim["claim_id"]]))
    report(results)
    return results

Each model runs at the lowest reasoning level it offers:

| Model | From | Reasoning |
|---|---|---|
| MiMo V2.6 Pro | Xiaomi | Not set: it has no reasoning setting |
| GLM 5.3 Flash | Z.ai | `low`: it always reasons, and low is its lowest level |
| Kimi K3 | Moonshot AI | `low` |
| DeepSeek V4.1 Flash | DeepSeek | `low` |
| Nemotron 3 Ultra | Nvidia | `medium`: its lowest level |
| Mistral Medium 3.5 | Mistral | `none`: it offers only high or none |

In [50]:
mimo = run("xiaomi/mimo-v2.6-pro", None, claims, labels)

xiaomi/mimo-v2.6-pro:   0%|          | 0/20 [00:00<?, ?it/s]

xiaomi/mimo-v2.6-pro: 16 of 20 decisions right, 14 with the payout too, $0.0170 in 364 seconds


In [51]:
glm = run("z-ai/glm-5.3-flash", "low", claims, labels, ignore=["wafer"])

z-ai/glm-5.3-flash:   0%|          | 0/20 [00:00<?, ?it/s]

z-ai/glm-5.3-flash: 16 of 20 decisions right, 13 with the payout too, $0.0040 in 66 seconds


In [52]:
kimi = run("moonshotai/kimi-k3", "low", claims, labels)

moonshotai/kimi-k3:   0%|          | 0/20 [00:00<?, ?it/s]

moonshotai/kimi-k3: 16 of 20 decisions right, 15 with the payout too, $0.0711 in 73 seconds


In [53]:
deepseek = run("deepseek/deepseek-v4.1-flash", "low", claims, labels)

deepseek/deepseek-v4.1-flash:   0%|          | 0/20 [00:00<?, ?it/s]

deepseek/deepseek-v4.1-flash: 17 of 20 decisions right, 16 with the payout too, $0.0118 in 122 seconds


In [54]:
nemotron = run("nvidia/nemotron-3-ultra-550b-a55b", "medium", claims, labels)

nvidia/nemotron-3-ultra-550b-a55b:   0%|          | 0/20 [00:00<?, ?it/s]

nvidia/nemotron-3-ultra-550b-a55b: 15 of 20 decisions right, 12 with the payout too, $0.0223 in 49 seconds


In [55]:
mistral = run("mistralai/mistral-medium-3-5", "none", claims, labels)

mistralai/mistral-medium-3-5:   0%|          | 0/20 [00:00<?, ?it/s]

mistralai/mistral-medium-3-5: 14 of 20 decisions right, 11 with the payout too, $0.0527 in 32 seconds


In [56]:
openai = run("openai/gpt-oss-120b", "medium", claims, labels)

openai/gpt-oss-120b:   0%|          | 0/20 [00:00<?, ?it/s]

openai/gpt-oss-120b: 12 of 20 decisions right, 11 with the payout too, $0.0031 in 125 seconds


In [66]:
def decide_locally(claim, model):
    """Ask a model on this computer to decide one claim through Ollama."""
    start = time.time()
    response = ollama_client.chat(
        model=model,
        messages=make_messages(claim),
        format=ClaimDecision.model_json_schema(),
        think=False,
        options={"num_ctx": 8192},
    )
    answer = ClaimDecision.model_validate_json(response.message.content)
    return {
        "model": model,
        "effort": "none",
        "where": "This computer",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "input_tokens": response.prompt_eval_count,
        "cost": 0.0,
        "seconds": round(time.time() - start -1),
    }

In [67]:
def run_locally(model, claims, labels):
    """Run a model on this computer on the claims and compare against the labels."""
    results = []
    for claim in tqdm(claims, desc=model):
        answer = decide_locally(claim, model)
        results.append(score(answer, labels[claim["claim_id"]]))
    report(results)
    return results

In [68]:
first_local = decide_locally(claims[0], "qwen3.5:4b")
first_local

{'model': 'qwen3.5:4b',
 'effort': 'none',
 'where': 'This computer',
 'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'rationale': "The incident occurred within the trip dates (2026-06-10 to 2026-06-18). The reason for the delay was a crew problem, which is covered under 'mechanical or crew problem'. All required documents (carrier confirmation and receipts) are attached. The claim amount ($420) includes accommodation ($245), meals ($95), and transport ($80). Since this is a 'Premium' plan, the Trip Delay waiting period is 6 hours (criterion met as wait was >6 hours) and the maximum payable amount is $1,500. Under the Payment section: No carrier payments are mentioned, and no deductible applies for Trip Delay claims. Therefore, the full claimed amount of $420 is covered.",
 'input_tokens': 1139,
 'cost': 0.0,
 'seconds': 6}

In [69]:
qwen = run_locally("qwen3.5:4b", claims, labels)

qwen3.5:4b:   0%|          | 0/20 [00:00<?, ?it/s]

qwen3.5:4b: 12 of 20 decisions right, 7 with the payout too, $0.0000 in 80 seconds


In [70]:
qwen = run_locally("gemma4:e2b-it-qat", claims, labels)

gemma4:e2b-it-qat:   0%|          | 0/20 [00:00<?, ?it/s]

gemma4:e2b-it-qat: 13 of 20 decisions right, 5 with the payout too, $0.0000 in 23 seconds


In [71]:
ollama_client.generate(model="qwen3.5:4b", keep_alive=0)

GenerateResponse(model='qwen3.5:4b', created_at='2026-10-30T18:41:57.8782185Z', done=True, done_reason='unload', total_duration=None, load_duration=None, prompt_eval_count=None, prompt_eval_duration=None, eval_count=None, eval_duration=None, response='', thinking=None, context=None, logprobs=None, image=None, completed=None, total=None)

In [72]:
ollama_client.generate(model="gemma4:e2b-it-qat", keep_alive=0)

GenerateResponse(model='gemma4:e2b-it-qat', created_at='2026-10-30T18:42:32.7392398Z', done=True, done_reason='unload', total_duration=None, load_duration=None, prompt_eval_count=None, prompt_eval_duration=None, eval_count=None, eval_duration=None, response='', thinking=None, context=None, logprobs=None, image=None, completed=None, total=None)

In [73]:
openai = run_locally("gpt-oss:20b", claims, labels)

gpt-oss:20b:   0%|          | 0/20 [00:00<?, ?it/s]

gpt-oss:20b: 17 of 20 decisions right, 13 with the payout too, $0.0000 in 763 seconds


In [74]:
ollama_client.generate(model="gpt-oss:20b", keep_alive=0)

GenerateResponse(model='gpt-oss:20b', created_at='2026-10-30T18:59:35.6071022Z', done=True, done_reason='unload', total_duration=None, load_duration=None, prompt_eval_count=None, prompt_eval_duration=None, eval_count=None, eval_duration=None, response='', thinking=None, context=None, logprobs=None, image=None, completed=None, total=None)

In [75]:
CLAIMS = 20

# model, where, decisions right, decision + payout right, total cost ($), total seconds
runs = [
    ("mimo-v2.6-pro",                 "OpenRouter",    16, 14, 0.0170, 364),
    ("glm-5.3-flash",                 "OpenRouter",    16, 13, 0.0040,  66),
    ("kimi-k3",                       "OpenRouter",    16, 15, 0.0711,  73),
    ("deepseek-v4.1-flash",           "OpenRouter",    17, 16, 0.0118, 122),
    ("nemotron-3-ultra-550b-a55b",    "OpenRouter",    15, 12, 0.0223,  49),
    ("mistral-medium-3-5",            "OpenRouter",    14, 11, 0.0527,  32),
    ("gpt-oss-120b",                  "OpenRouter",    12, 11, 0.0031, 125),
    ("qwen3.5:4b",                    "This computer", 12,  7, 0.0,     80),
    ("gemma4:e2b-it-qat",             "This computer", 13,  5, 0.0,     23),
    ("gpt-oss:20b",                   "This computer", 17, 13, 0.0,    763),
]

summary = pd.DataFrame(runs, columns=["model", "where", "decisions", "both", "cost", "seconds"])

summary["decision right (%)"] = round(100 * summary["decisions"] / CLAIMS)
summary["decision and payout right (%)"] = round(100 * summary["both"] / CLAIMS)
summary["seconds a claim"] = round(summary["seconds"] / CLAIMS, 1)
summary["cost a claim ($)"] = round(summary["cost"] / CLAIMS, 5)

summary = summary.drop(columns=["decisions", "both", "cost", "seconds"])
summary

,model,where,decision right (%),decision and payout right (%),seconds a claim,cost a claim ($)
0,mimo-v2.6-pro,OpenRouter,80.0,70.0,18.2,0.00085
1,glm-5.3-flash,OpenRouter,80.0,65.0,3.3,0.00020
2,kimi-k3,OpenRouter,80.0,75.0,3.6,0.00356
3,deepseek-v4.1-flash,OpenRouter,85.0,80.0,6.1,0.00059
4,nemotron-3-ultra-550b-a55b,OpenRouter,75.0,60.0,2.4,0.00112
5,mistral-medium-3-5,OpenRouter,70.0,55.0,1.6,0.00264
6,gpt-oss-120b,OpenRouter,60.0,55.0,6.2,0.00016
7,qwen3.5:4b,This computer,60.0,35.0,4.0,0.00000
8,gemma4:e2b-it-qat,This computer,65.0,25.0,1.2,0.00000
9,gpt-oss:20b,This computer,85.0,65.0,38.2,0.00000


In [76]:
# Accuracy vs. Cost

In [77]:
fig = px.scatter(
    summary,
    x="cost a claim ($)",
    y="decision and payout right (%)",
    color="where",
    text="model",
    title="Accuracy against cost",
    height=600,
)
fig.update_traces(marker_size=14, textposition="top center")
fig.update_yaxes(range=[0, 100])
fig.show()

In [78]:
fig = px.scatter(
    summary,
    x="seconds a claim",
    y="decision and payout right (%)",
    color="where",
    text="model",
    title="Accuracy against speed",
    height=600,
)
fig.update_traces(marker_size=14, textposition="top center")
fig.update_yaxes(range=[0, 100])
fig.show()

In [79]:
fig = px.scatter_3d(
    summary,
    x="cost a claim ($)",
    y="seconds a claim",
    z="decision and payout right (%)",
    color="where",
    text="model",
    title="Accuracy, cost and speed",
    height=700,
)
fig.show()

### Overall considerations - accuracy, cost, latency - understand budget, cost of innacurate results, implication of time bound results